In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])

0

In [2]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Transformers: 4.52.4
Accelerate: 1.7.0
Datasets: 3.6.0
Torch: 2.7.1+cu126
CUDA available: True


In [4]:
# =========================================================
# 1. IMPORTS AND GOOGLE DRIVE SETUP
# =========================================================
import os
import csv
import time
import random

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset
from scipy.stats import pearsonr
from google.colab import drive

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)

from sklearn.metrics import f1_score


drive.mount("/content/drive")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# =========================================================
# 2. OUTPUT PATHS
# =========================================================
BASE_DIR = "/content/drive/MyDrive"

EPOCH_SENTENCE_DIR = os.path.join(
    BASE_DIR,
    "RoBERTa_SingleStep_Epoch_Test_Logs"
)

EPOCH_METRICS_FILE = os.path.join(
    BASE_DIR,
    "RoBERTa_SingleStep_Epoch_Metrics.csv"
)

os.makedirs(
    EPOCH_SENTENCE_DIR,
    exist_ok=True
)


# Create/reset the epoch metrics file
with open(
    EPOCH_METRICS_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "epoch",
        "train_loss",
        "validation_loss",
        "validation_f1_macro",
        "validation_f1_micro",
        "validation_pearson_mean"
    ])


# =========================================================
# 3. CONFIGURATION
# =========================================================
SEED = 42
THRESHOLD = 0.5

EMOTIONS = [
    "anger",
    "fear",
    "joy",
    "sadness",
    "surprise"
]

LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]

NUM_LABELS = len(LABELS)


# =========================================================
# 4. SEED SETUP
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)


# =========================================================
# 5. LOAD BRIGHTER DATASET
# =========================================================
print("\nLoading BRIGHTER dataset...")

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("\nOriginal split sizes:")

print({
    "train": len(train_df),
    "validation": len(val_df),
    "test": len(test_df)
})


# =========================================================
# 6. CREATE SINGLE-STEP LABELS
# =========================================================
def convert_single_step_labels(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(
            columns=["disgust"]
        )

    for emotion in EMOTIONS:
        for level in LEVELS:
            df[f"{emotion}_{level}"] = (
                df[emotion] == level
            ).astype(int)

    return df


train_single_df = convert_single_step_labels(
    train_df
)

val_single_df = convert_single_step_labels(
    val_df
)

test_single_df = convert_single_step_labels(
    test_df
)

print("\nSingle-step sample:")

display(
    train_single_df[
        ["text"] + LABELS
    ].head()
)


# =========================================================
# 7. CREATE 70/20/10 SPLIT
# =========================================================
full_df = pd.concat(
    [
        train_single_df,
        val_single_df,
        test_single_df
    ],
    ignore_index=True
)

full_df = full_df[
    ["text"] + LABELS
]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

total_samples = len(full_df)

train_end = int(
    0.70 * total_samples
)

validation_end = int(
    0.90 * total_samples
)

train_split_df = full_df[
    :train_end
].reset_index(drop=True)

val_split_df = full_df[
    train_end:validation_end
].reset_index(drop=True)

test_split_df = full_df[
    validation_end:
].reset_index(drop=True)

print("\nNew split sizes:")

print({
    "train": len(train_split_df),
    "validation": len(val_split_df),
    "test": len(test_split_df)
})


# Keep the original test sentences
test_texts = test_split_df[
    "text"
].tolist()


# =========================================================
# 8. CONVERT TO HUGGING FACE DATASETS
# =========================================================
train_single = Dataset.from_pandas(
    train_split_df,
    preserve_index=False
)

val_single = Dataset.from_pandas(
    val_split_df,
    preserve_index=False
)

test_single = Dataset.from_pandas(
    test_split_df,
    preserve_index=False
)


# =========================================================
# 9. TOKENIZATION
# =========================================================
tokenizer = RobertaTokenizer.from_pretrained(
    "roberta-base"
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )


train_single = train_single.map(
    tokenize_function,
    batched=True
)

val_single = val_single.map(
    tokenize_function,
    batched=True
)

test_single = test_single.map(
    tokenize_function,
    batched=True
)


# =========================================================
# 10. ADD LABEL VECTORS
# =========================================================
def add_labels(example):
    example["labels"] = [
        float(example[label])
        for label in LABELS
    ]

    return example


train_single = train_single.map(
    add_labels
)

val_single = val_single.map(
    add_labels
)

test_single = test_single.map(
    add_labels
)


train_single.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

val_single.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_single.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)


# =========================================================
# 11. VALIDATION METRICS
# =========================================================
def compute_metrics(eval_prediction):
    logits, true_labels = eval_prediction

    probabilities = 1 / (
        1 + np.exp(-logits)
    )

    predicted_labels = (
        probabilities >= THRESHOLD
    ).astype(int)

    f1_macro = f1_score(
        true_labels,
        predicted_labels,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        true_labels,
        predicted_labels,
        average="micro",
        zero_division=0
    )

    pearson_scores = []

    for label_index in range(
        true_labels.shape[1]
    ):
        true_column = true_labels[
            :,
            label_index
        ]

        probability_column = probabilities[
            :,
            label_index
        ]

        if (
            np.std(true_column) == 0
            or
            np.std(probability_column) == 0
        ):
            pearson_scores.append(0.0)

        else:
            correlation, _ = pearsonr(
                true_column,
                probability_column
            )

            if np.isnan(correlation):
                correlation = 0.0

            pearson_scores.append(
                float(correlation)
            )

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": float(
            np.mean(pearson_scores)
        )
    }


# =========================================================
# 12. HELPER FUNCTIONS
# =========================================================
def labels_to_text(
    binary_labels,
    label_names
):
    selected_labels = [
        label_names[index]
        for index, value
        in enumerate(binary_labels)
        if int(value) == 1
    ]

    if len(selected_labels) == 0:
        return "No Emotion"

    return ", ".join(
        selected_labels
    )


def clean_probability(value):
    value = round(
        float(value),
        4
    )

    if value == 0:
        return 0

    if value == 1:
        return 1

    return value


# =========================================================
# 13. CALLBACK: SAVE TEST SENTENCES AFTER EACH EPOCH
# =========================================================
class SaveEpochTestSentencesCallback(
    TrainerCallback
):

    def __init__(
        self,
        test_dataset,
        test_texts,
        sentence_output_dir,
        metrics_output_file
    ):
        self.test_dataset = test_dataset
        self.test_texts = test_texts

        self.sentence_output_dir = (
            sentence_output_dir
        )

        self.metrics_output_file = (
            metrics_output_file
        )

        self.trainer_ref = None
        self.current_train_loss = None

    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs
    ):
        if (
            logs is not None
            and "loss" in logs
            and "eval_loss" not in logs
        ):
            self.current_train_loss = float(
                logs["loss"]
            )

    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        **kwargs
    ):
        if metrics is None:
            return

        # Run only after validation evaluation
        if "eval_loss" not in metrics:
            return

        epoch = int(
            round(
                float(
                    metrics.get(
                        "epoch",
                        state.epoch
                    )
                )
            )
        )

        validation_loss = float(
            metrics.get(
                "eval_loss",
                0.0
            )
        )

        validation_f1_macro = float(
            metrics.get(
                "eval_f1_macro",
                0.0
            )
        )

        validation_f1_micro = float(
            metrics.get(
                "eval_f1_micro",
                0.0
            )
        )

        validation_pearson = float(
            metrics.get(
                "eval_pearson_mean",
                0.0
            )
        )

        train_loss = (
            self.current_train_loss
            if self.current_train_loss
            is not None
            else ""
        )

        # Save one validation-metrics row
        with open(
            self.metrics_output_file,
            "a",
            newline="",
            encoding="utf-8"
        ) as file:

            writer = csv.writer(file)

            writer.writerow([
                epoch,
                train_loss,
                validation_loss,
                validation_f1_macro,
                validation_f1_micro,
                validation_pearson
            ])

        # Predict the complete test set
        prediction_output = (
            self.trainer_ref.predict(
                self.test_dataset
            )
        )

        logits = (
            prediction_output.predictions
        )

        true_labels = (
            prediction_output.label_ids
            .astype(int)
        )

        probabilities = 1 / (
            1 + np.exp(-logits)
        )

        predicted_labels = (
            probabilities >= THRESHOLD
        ).astype(int)

        sentence_rows = []

        for sentence_id in range(
            len(self.test_texts)
        ):
            row = {
                "epoch": epoch,

                "sentence_id":
                    sentence_id,

                "sentence":
                    self.test_texts[
                        sentence_id
                    ],

                "true_labels":
                    labels_to_text(
                        true_labels[
                            sentence_id
                        ],
                        LABELS
                    ),

                "predicted_labels":
                    labels_to_text(
                        predicted_labels[
                            sentence_id
                        ],
                        LABELS
                    )
            }

            for label_index, label in enumerate(
                LABELS
            ):
                row[
                    f"prob_{label}"
                ] = clean_probability(
                    probabilities[
                        sentence_id,
                        label_index
                    ]
                )

            sentence_rows.append(row)

        epoch_sentence_df = pd.DataFrame(
            sentence_rows
        )

        epoch_file = os.path.join(
            self.sentence_output_dir,
            (
                "RoBERTa_SingleStep_"
                f"Test_Epoch_{epoch}.csv"
            )
        )

        epoch_sentence_df.to_csv(
            epoch_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"\nTest sentence log saved "
            f"for epoch {epoch}:"
        )

        print(epoch_file)


# =========================================================
# 14. MODEL
# =========================================================
model = (
    RobertaForSequenceClassification
    .from_pretrained(
        "roberta-base",
        num_labels=NUM_LABELS,
        problem_type=(
            "multi_label_classification"
        )
    )
)


# =========================================================
# 15. TRAINING ARGUMENTS
# =========================================================
training_args = TrainingArguments(
    output_dir="/content/roberta_output",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=10,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=SEED
)


# =========================================================
# 16. TRAINER
# =========================================================
epoch_callback = (
    SaveEpochTestSentencesCallback(
        test_dataset=test_single,
        test_texts=test_texts,

        sentence_output_dir=(
            EPOCH_SENTENCE_DIR
        ),

        metrics_output_file=(
            EPOCH_METRICS_FILE
        )
    )
)


trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_single,
    eval_dataset=val_single,

    data_collator=data_collator,
    processing_class=tokenizer,

    compute_metrics=compute_metrics,

    callbacks=[
        epoch_callback,
         EarlyStoppingCallback(
                    early_stopping_patience=1,
                    early_stopping_threshold=0.0
                )
    ]
)

epoch_callback.trainer_ref = trainer


# =========================================================
# 17. TRAIN MODEL
# =========================================================
start_time = time.time()

trainer.train()

end_time = time.time()

print(
    "\nTotal training time:",
    round(
        end_time - start_time,
        1
    ),
    "seconds"
)





Using device: cuda

Loading BRIGHTER dataset...

Original split sizes:
{'train': 2763, 'validation': 115, 'test': 2765}

Single-step sample:


,text,anger_1,anger_2,anger_3,fear_1,fear_2,fear_3,joy_1,joy_2,joy_3,sadness_1,sadness_2,sadness_3,surprise_1,surprise_2,surprise_3
0,"Colorado, middle of nowhere.",0,0,0,1,0,0,0,0,0,0,0,0,1,0,0
1,This involved swimming a pretty large lake tha...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,It was one of my most shameful experiences.,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
3,"After all, I had vegetables coming out my ears...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Then the screaming started.,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0



New split sizes:
{'train': 3950, 'validation': 1128, 'test': 565}


Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Pearson Mean
1,0.308600,0.270475,0.024440,0.051585,0.280611
2,0.250800,0.245216,0.127966,0.177632,0.411228
3,0.214300,0.235482,0.188426,0.253705,0.440191
4,0.186700,0.230233,0.282333,0.380512,0.461322
5,0.160900,0.235222,0.325802,0.409950,0.467977



Test sentence log saved for epoch 1:
/content/drive/MyDrive/RoBERTa_SingleStep_Epoch_Test_Logs/RoBERTa_SingleStep_Test_Epoch_1.csv

Test sentence log saved for epoch 2:
/content/drive/MyDrive/RoBERTa_SingleStep_Epoch_Test_Logs/RoBERTa_SingleStep_Test_Epoch_2.csv

Test sentence log saved for epoch 3:
/content/drive/MyDrive/RoBERTa_SingleStep_Epoch_Test_Logs/RoBERTa_SingleStep_Test_Epoch_3.csv

Test sentence log saved for epoch 4:
/content/drive/MyDrive/RoBERTa_SingleStep_Epoch_Test_Logs/RoBERTa_SingleStep_Test_Epoch_4.csv

Test sentence log saved for epoch 5:
/content/drive/MyDrive/RoBERTa_SingleStep_Epoch_Test_Logs/RoBERTa_SingleStep_Test_Epoch_5.csv

Total training time: 305.0 seconds

Epoch validation metrics:
/content/drive/MyDrive/RoBERTa_SingleStep_Epoch_Metrics.csv

Epoch test sentence files:
/content/drive/MyDrive/RoBERTa_SingleStep_Epoch_Test_Logs

Validation results for choosing the best epoch:


,epoch,train_loss,validation_loss,validation_f1_macro,validation_f1_micro,validation_pearson_mean
0,1,0.3086,0.2705,0.0244,0.0516,0.2806
1,2,0.2508,0.2452,0.1280,0.1776,0.4112
2,3,0.2143,0.2355,0.1884,0.2537,0.4402
3,4,0.1867,0.2302,0.2823,0.3805,0.4613
4,5,0.1609,0.2352,0.3258,0.4100,0.4680
